# Week 38, Day 4: Project: interface

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w038d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('errors.py', 'w', encoding='utf-8') as f:
    f.write('"""Every error the library raises on purpose."""\n\n\nclass LibraryError(Exception):\n    """Base class for the library\'s own errors."""\n\n\nclass InvalidInput(LibraryError, ValueError):\n    """Something typed or loaded doesn\'t make sense."""\n\n\nclass NotFound(LibraryError, LookupError):\n    """There\'s no book or member with that id."""\n\n\nclass NotAllowed(LibraryError):\n    """The library\'s rules don\'t allow this right now."""\n\n\nclass StorageError(LibraryError):\n    """The library\'s data couldn\'t be read or saved."""\n')
with open('values.py', 'w', encoding='utf-8') as f:
    f.write('"""Reading and checking the values people type, and showing money."""\nfrom errors import InvalidInput\n\nKINDS = {"B": "book", "M": "member"}\n\n\ndef clean_isbn(text):\n    """\'978-0-14-118776-1\' as \'9780141187761\'. Raises InvalidInput."""\n    shown = text.strip()\n    digits = shown.replace("-", "").replace(" ", "")\n    if len(digits) != 13 or not (digits.isascii() and digits.isdigit()):\n        raise InvalidInput(f"an ISBN has 13 digits: {shown!r}")\n    total = sum(int(d) * (3 if i % 2 else 1) for i, d in enumerate(digits))\n    if total % 10 != 0:\n        raise InvalidInput(f"wrong check digit: {shown!r}")\n    return digits\n\n\ndef clean_text(text, what, longest):\n    """\'  Things   Fall Apart \' as \'Things Fall Apart\'. Raises InvalidInput."""\n    cleaned = " ".join(text.split())\n    if not cleaned:\n        raise InvalidInput(f"{what} is missing")\n    if len(cleaned) > longest:\n        raise InvalidInput(f"{what} is longer than {longest} characters")\n    return cleaned\n\n\ndef parse_id(text, kind):\n    """\' b12 \' as \'B12\', for kind \'B\'. Raises InvalidInput."""\n    cleaned = text.strip().upper()\n    number = cleaned[1:]\n    if not cleaned.startswith(kind) or not (number.isascii() and number.isdigit()) or number.startswith("0"):\n        raise InvalidInput(f"not a {KINDS[kind]} id: {text.strip()!r}")\n    return cleaned\n\n\ndef parse_pence(text):\n    """\'£1.20\' or \'0.6\' as whole pence. Raises InvalidInput."""\n    shown = text.strip()\n    pounds, dot, pence = shown.removeprefix("£").partition(".")\n    if (not (pounds.isascii() and pounds.isdigit())\n            or (dot and not (pence.isascii() and pence.isdigit() and len(pence) <= 2))):\n        raise InvalidInput(f"not an amount: {shown!r}")\n    return int(pounds) * 100 + (int(pence.ljust(2, "0")) if dot else 0)\n\n\ndef format_pence(pence):\n    """60 as \'£0.60\', -350 as \'-£3.50\'."""\n    sign = "-" if pence < 0 else ""\n    pounds, rest = divmod(abs(pence), 100)\n    return f"{sign}£{pounds:,}.{rest:02d}"\n\n\ndef check_all(**checks):\n    """Run each check (a function of no arguments); return their results by name, or raise one\n    InvalidInput listing every problem, in order."""\n    results, problems = {}, []\n    for name, check in checks.items():\n        try:\n            results[name] = check()\n        except InvalidInput as err:\n            problems.append(str(err))\n    if problems:\n        raise InvalidInput("; ".join(problems))\n    return results\n')
with open('domain.py', 'w', encoding='utf-8') as f:
    f.write('"""The library\'s things and rules. No input, output or storage here."""\nfrom dataclasses import dataclass, field\nfrom datetime import date, timedelta\n\nfrom errors import InvalidInput, NotAllowed\nfrom values import format_pence\n\nLOAN_DAYS = 21\nFINE_PER_DAY = 20      # pence for each day late\nMAX_FINE = 500         # the most one loan can cost\nFINE_LIMIT = 500       # owing this much stops a member borrowing\n\n\n@dataclass\nclass Book:\n    book_id: str\n    isbn: str\n    title: str\n    author: str\n    borrower: str | None = None\n    holds: list[str] = field(default_factory=list)\n\n    @property\n    def on_loan(self):\n        return self.borrower is not None\n\n    def describe(self):\n        return f"{self.book_id}: {self.title} by {self.author}"\n\n\n@dataclass\nclass Member:\n    member_id: str\n    name: str\n    max_loans: int = 3\n    owes: int = 0\n\n\n@dataclass\nclass Loan:\n    loan_id: int\n    book_id: str\n    member_id: str\n    out: date\n    due: date\n    returned: date | None = None\n\n    @property\n    def is_open(self):\n        return self.returned is None\n\n    def days_late(self, today):\n        end = self.returned or today\n        return max(0, (end - self.due).days)\n\n    def fine(self, today):\n        return min(self.days_late(today) * FINE_PER_DAY, MAX_FINE)\n\n\ndef check_can_borrow(member, book, open_loans, today):\n    """Raise NotAllowed with the first reason the member can\'t borrow the book today."""\n    if book.on_loan:\n        raise NotAllowed(f"{book.book_id} is on loan")\n    if book.holds and book.holds[0] != member.member_id:\n        raise NotAllowed(f"{book.book_id} is being kept for someone else")\n    if len(open_loans) >= member.max_loans:\n        raise NotAllowed(f"{member.name} already has {len(open_loans)} books")\n    if any(loan.days_late(today) for loan in open_loans):\n        raise NotAllowed(f"{member.name} has an overdue book")\n    if member.owes >= FINE_LIMIT:\n        raise NotAllowed(f"{member.name} owes {format_pence(member.owes)}")\n\n\ndef lend(book, member, open_loans, today, loan_id):\n    """Lend the book, or raise NotAllowed. Returns the new Loan."""\n    check_can_borrow(member, book, open_loans, today)\n    if book.holds:\n        book.holds.pop(0)\n    book.borrower = member.member_id\n    return Loan(loan_id, book.book_id, member.member_id, today, today + timedelta(days=LOAN_DAYS))\n\n\ndef give_back(book, loan, member, today):\n    """Close the loan and add any fine to what the member owes. Returns the fine."""\n    assert loan.is_open and loan.book_id == book.book_id and loan.member_id == member.member_id\n    loan.returned = today\n    book.borrower = None\n    fine = loan.fine(today)\n    member.owes += fine\n    return fine\n\n\ndef reserve(book, member):\n    """Join the queue for a book. Returns the member\'s place in it."""\n    if not book.on_loan and not book.holds:\n        raise NotAllowed(f"{book.book_id} is on the shelf: borrow it instead")\n    if book.borrower == member.member_id:\n        raise NotAllowed(f"{member.name} already has {book.book_id}")\n    if member.member_id in book.holds:\n        raise NotAllowed(f"{member.name} is already waiting for {book.book_id}")\n    book.holds.append(member.member_id)\n    return len(book.holds)\n\n\ndef pay(member, pence):\n    """Take a payment towards the member\'s fines. Returns what they still owe."""\n    if pence <= 0:\n        raise InvalidInput("a payment must be more than £0")\n    if pence > member.owes:\n        raise NotAllowed(f"{member.name} only owes {format_pence(member.owes)}")\n    member.owes -= pence\n    return member.owes\n')
with open('store.py', 'w', encoding='utf-8') as f:
    f.write('"""Where the library\'s data is kept: in memory, or in a JSON file."""\nimport json\nimport logging\nimport os\nfrom dataclasses import asdict\nfrom datetime import date\n\nfrom domain import Book, Loan, Member\nfrom errors import NotFound, StorageError\n\nlog = logging.getLogger("shelf.store")\n\n\nclass MemoryStore:\n    """Keeps everything in dictionaries. commit() does nothing: there\'s nowhere to save."""\n\n    def __init__(self):\n        self.books = {}\n        self.members = {}\n        self.loans = []\n        self.counters = {"B": 0, "M": 0, "L": 0}\n\n    def next_id(self, kind):\n        """\'B1\', \'B2\', ... for books, \'M1\', ... for members, and 1, 2, ... for loans. Never reused."""\n        self.counters[kind] += 1\n        number = self.counters[kind]\n        return number if kind == "L" else f"{kind}{number}"\n\n    def add_book(self, book):\n        self.books[book.book_id] = book\n\n    def add_member(self, member):\n        self.members[member.member_id] = member\n\n    def add_loan(self, loan):\n        self.loans.append(loan)\n\n    def book(self, book_id):\n        try:\n            return self.books[book_id]\n        except KeyError:\n            raise NotFound(f"no book {book_id}") from None\n\n    def member(self, member_id):\n        try:\n            return self.members[member_id]\n        except KeyError:\n            raise NotFound(f"no member {member_id}") from None\n\n    def open_loans(self, member_id=None):\n        return [loan for loan in self.loans\n                if loan.is_open and (member_id is None or loan.member_id == member_id)]\n\n    def open_loan_for(self, book_id):\n        for loan in self.loans:\n            if loan.is_open and loan.book_id == book_id:\n                return loan\n        raise NotFound(f"{book_id} isn\'t on loan")\n\n    def commit(self):\n        pass\n\n\ndef to_data(store):\n    """Everything in the store as plain dictionaries and lists, ready for JSON."""\n    def loan_data(loan):\n        d = asdict(loan)\n        d["out"], d["due"] = loan.out.isoformat(), loan.due.isoformat()\n        d["returned"] = loan.returned.isoformat() if loan.returned else None\n        return d\n    return {\n        "books": [asdict(b) for b in store.books.values()],\n        "members": [asdict(m) for m in store.members.values()],\n        "loans": [loan_data(loan) for loan in store.loans],\n        "counters": dict(store.counters),\n    }\n\n\ndef fill(store, data):\n    """Load data made by to_data into an empty store."""\n    for d in data["books"]:\n        store.add_book(Book(**d))\n    for d in data["members"]:\n        store.add_member(Member(**d))\n    for d in data["loans"]:\n        d = dict(d)\n        d["out"], d["due"] = date.fromisoformat(d["out"]), date.fromisoformat(d["due"])\n        d["returned"] = date.fromisoformat(d["returned"]) if d["returned"] else None\n        store.add_loan(Loan(**d))\n    store.counters = dict(data["counters"])\n\n\nclass JsonStore(MemoryStore):\n    """A MemoryStore that loads from a JSON file when it\'s made, and saves on commit()."""\n\n    def __init__(self, path):\n        super().__init__()\n        self.path = path\n        self._load()\n\n    def _load(self):\n        try:\n            with open(self.path, encoding="utf-8") as f:\n                data = json.load(f)\n        except FileNotFoundError:\n            log.info("no file at %s yet: starting empty", self.path)\n            return\n        except (OSError, ValueError) as err:\n            raise StorageError(f"can\'t read {self.path}") from err\n        try:\n            fill(self, data)\n        except (KeyError, TypeError, ValueError) as err:\n            raise StorageError(f"{self.path} is damaged") from err\n        log.info("loaded %d books, %d members and %d loans from %s",\n                 len(self.books), len(self.members), len(self.loans), self.path)\n\n    def commit(self):\n        """Save everything safely: write a temporary file, then replace the old file in one step."""\n        temp = self.path + ".tmp"\n        try:\n            with open(temp, "w", encoding="utf-8") as f:\n                json.dump(to_data(self), f, indent=1, ensure_ascii=False)\n            os.replace(temp, self.path)\n        except OSError as err:\n            raise StorageError(f"can\'t save to {self.path}") from err\n        finally:\n            if os.path.exists(temp):\n                os.remove(temp)\n        log.debug("saved %s", self.path)\n')

### Exercise: Step 19: books and members

Write the start of the service, `Library(store, today)`, where `today` is a function returning today's date. Give it:

- `book(book_id)` and `member(member_id)`: read the id with `parse_id` (so `" b1 "` works) and return the object from the store;
- `add_book(isbn, title, author)`: check all three with `check_all` (ISBN with `clean_isbn`, title up to 100 characters, author up to 60), then make a `Book` with the next `"B"` id, add it, **commit**, log `INFO` `"added B1"`, and return the book;
- `join(name)`: the same for a `Member` (name up to 60 characters), logging `"added M1"`.

In [ ]:
import logging

from domain import Book, Member, check_can_borrow, give_back, lend, pay, reserve
from values import check_all, clean_isbn, clean_text, parse_id, parse_pence

log = logging.getLogger("shelf.service")


class Library:
    def __init__(self, store, today):
        self.store = store
        self.today = today          # a function returning today's date

    def book(self, book_id):
        return self.store.book(parse_id(book_id, "B"))

    def member(self, member_id):
        return self.store.member(parse_id(member_id, "M"))
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nfrom errors import InvalidInput, NotAllowed, NotFound\nfrom domain import Book, Member\nfrom store import MemoryStore as _Mem\nD = _dt.date\nclass _Counting(_Mem):\n    def __init__(self):\n        super().__init__()\n        self.commits = 0\n    def commit(self):\n        self.commits += 1\nDAY = [D(2026, 11, 2)]\ndef make():\n    DAY[0] = D(2026, 11, 2)\n    return Library(_Counting(), lambda: DAY[0])\ndef refused(call, error, message):\n    try:\n        call()\n    except error as err:\n        assert str(err) == message, f\"got {str(err)!r}, expected {message!r}\"\n        return\n    raise AssertionError(f\"expected {error.__name__}: {message}\")\n\ndef test_lookups():\n    \"finding by id\"\n    lib = make()\n    lib.store.add_book(Book(\"B1\", \"9780141187761\", \"T\", \"A\"))\n    lib.store.add_member(Member(\"M1\", \"Ada\"))\n    assert lib.book(\" b1 \").title == \"T\" and lib.member(\"m1\").name == \"Ada\"\n    refused(lambda: lib.book(\"x\"), InvalidInput, \"not a book id: 'x'\")\n    refused(lambda: lib.member(\"M7\"), NotFound, \"no member M7\")\ndef test_add():\n    \"adding\"\n    lib = make()\n    b = lib.add_book(\" 978-0-14-118776-1\", \"Things  Fall Apart \", \"Chinua Achebe\")\n    assert b == Book(\"B1\", \"9780141187761\", \"Things Fall Apart\", \"Chinua Achebe\") and lib.store.book(\"B1\") is b\n    m = lib.join(\"  Ada   Obi \")\n    assert m == Member(\"M1\", \"Ada Obi\") and lib.store.member(\"M1\") is m\n    assert lib.store.commits == 2, \"commit after each change\"\n    refused(lambda: lib.add_book(\"123\", \"\", \"x\" * 61), InvalidInput, \"an ISBN has 13 digits: '123'; title is missing; author is longer than 60 characters\")\n    refused(lambda: lib.join(\" \"), InvalidInput, \"name is missing\")\n    assert len(lib.store.books) == 1 and lib.store.commits == 2 and lib.add_book(\"9780435950828\", \"Arrow of God\", \"Chinua Achebe\").book_id == \"B2\"")

### Exercise: Step 20: lending and returning

Add two use cases to `Library`:

- `lend(book_id, member_id)`: find the book and member, check the rules with the domain's `check_can_borrow` **before** taking the next `"L"` id (so a refused loan uses up no id), then lend with the domain's `lend` (passing the member's open loans, today's date, and that id), add the loan to the store, commit, log `INFO` `"lent B1 to M1"`, and return the loan;
- `give_back(book_id)`: find the book and its open loan, give it back with the domain's `give_back` (the member is the loan's), commit, log `INFO` `"B1 returned, fine 60"`, and return `(loan, fine)`.

In [ ]:
import logging

from domain import Book, Member, check_can_borrow, give_back, lend, pay, reserve
from values import check_all, clean_isbn, clean_text, parse_id, parse_pence

log = logging.getLogger("shelf.service")


class Library:
    def __init__(self, store, today):
        self.store = store
        self.today = today          # a function returning today's date

    def book(self, book_id):
        return self.store.book(parse_id(book_id, "B"))

    def member(self, member_id):
        return self.store.member(parse_id(member_id, "M"))

    def add_book(self, isbn, title, author):
        v = check_all(isbn=lambda: clean_isbn(isbn),
                      title=lambda: clean_text(title, "title", 100),
                      author=lambda: clean_text(author, "author", 60))
        book = Book(self.store.next_id("B"), v["isbn"], v["title"], v["author"])
        self.store.add_book(book)
        self.store.commit()
        log.info("added %s", book.book_id)
        return book

    def join(self, name):
        member = Member(self.store.next_id("M"), clean_text(name, "name", 60))
        self.store.add_member(member)
        self.store.commit()
        log.info("added %s", member.member_id)
        return member

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nfrom errors import InvalidInput, NotAllowed, NotFound\nfrom domain import Book, Member\nfrom store import MemoryStore as _Mem\nD = _dt.date\nclass _Counting(_Mem):\n    def __init__(self):\n        super().__init__()\n        self.commits = 0\n    def commit(self):\n        self.commits += 1\nDAY = [D(2026, 11, 2)]\ndef make():\n    DAY[0] = D(2026, 11, 2)\n    return Library(_Counting(), lambda: DAY[0])\ndef refused(call, error, message):\n    try:\n        call()\n    except error as err:\n        assert str(err) == message, f\"got {str(err)!r}, expected {message!r}\"\n        return\n    raise AssertionError(f\"expected {error.__name__}: {message}\")\n\ndef stocked():\n    lib = make()\n    lib.add_book(\"978-0-14-118776-1\", \"Things Fall Apart\", \"Chinua Achebe\")\n    lib.add_book(\"9780435950828\", \"Arrow of God\", \"Chinua Achebe\")\n    lib.add_book(\"978-1-84749-058-2\", \"Half of a Yellow Sun\", \"Chimamanda Ngozi Adichie\")\n    lib.join(\"Ada Obi\")\n    lib.join(\"Bo Lee\")\n    lib.join(\"Chi Eze\")\n    return lib\n\ndef test_lend():\n    \"lending\"\n    lib = stocked()\n    loan = lib.lend(\"b1\", \" m1\")\n    assert (loan.loan_id, loan.book_id, loan.member_id, loan.due) == (1, \"B1\", \"M1\", D(2026, 11, 23))\n    assert lib.store.loans == [loan], \"add the loan to the store\"\n    assert lib.book(\"B1\").borrower == \"M1\"\n    assert lib.store.commits == 7, f\"commit after lending ({lib.store.commits} commits)\"\n    refused(lambda: lib.lend(\"B1\", \"M2\"), NotAllowed, \"B1 is on loan\")\n    assert lib.lend(\"B2\", \"M1\").loan_id == 2\ndef test_return():\n    \"returning\"\n    lib = stocked()\n    lib.lend(\"B1\", \"M1\")\n    DAY[0] = D(2026, 11, 26)\n    loan, fine = lib.give_back(\" b1\")\n    assert fine == 60 and loan.returned == D(2026, 11, 26) and lib.member(\"M1\").owes == 60 and not lib.book(\"B1\").on_loan\n    refused(lambda: lib.give_back(\"B1\"), NotFound, \"B1 isn't on loan\")\n    refused(lambda: lib.give_back(\"B9\"), NotFound, \"no book B9\")")

### Exercise: Step 21: queues, payments and questions

Add the rest of the use cases:

- `reserve(book_id, member_id)` and `pay(member_id, amount)` (the amount is text, read with `parse_pence`): call the domain's function, commit, and return its result;
- `overdue()`: a list of `(loan, days_late)` for every open loan that's late today, most days late first, then by book id;
- `find(words)`: the books whose title or author (in any case) contains **every** word in `words`, sorted by the number in their id (so `B2` comes before `B10`). Empty words find every book.

In [ ]:
import logging

from domain import Book, Member, check_can_borrow, give_back, lend, pay, reserve
from values import check_all, clean_isbn, clean_text, parse_id, parse_pence

log = logging.getLogger("shelf.service")


class Library:
    def __init__(self, store, today):
        self.store = store
        self.today = today          # a function returning today's date

    def book(self, book_id):
        return self.store.book(parse_id(book_id, "B"))

    def member(self, member_id):
        return self.store.member(parse_id(member_id, "M"))

    def add_book(self, isbn, title, author):
        v = check_all(isbn=lambda: clean_isbn(isbn),
                      title=lambda: clean_text(title, "title", 100),
                      author=lambda: clean_text(author, "author", 60))
        book = Book(self.store.next_id("B"), v["isbn"], v["title"], v["author"])
        self.store.add_book(book)
        self.store.commit()
        log.info("added %s", book.book_id)
        return book

    def join(self, name):
        member = Member(self.store.next_id("M"), clean_text(name, "name", 60))
        self.store.add_member(member)
        self.store.commit()
        log.info("added %s", member.member_id)
        return member

    def lend(self, book_id, member_id):
        book, member = self.book(book_id), self.member(member_id)
        open_loans, today = self.store.open_loans(member.member_id), self.today()
        check_can_borrow(member, book, open_loans, today)    # first, so a refusal uses up no loan id
        loan = lend(book, member, open_loans, today, self.store.next_id("L"))
        self.store.add_loan(loan)
        self.store.commit()
        log.info("lent %s to %s", book.book_id, member.member_id)
        return loan

    def give_back(self, book_id):
        book = self.book(book_id)
        loan = self.store.open_loan_for(book.book_id)
        fine = give_back(book, loan, self.store.member(loan.member_id), self.today())
        self.store.commit()
        log.info("%s returned, fine %d", book.book_id, fine)
        return loan, fine

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nfrom errors import InvalidInput, NotAllowed, NotFound\nfrom domain import Book, Member\nfrom store import MemoryStore as _Mem\nD = _dt.date\nclass _Counting(_Mem):\n    def __init__(self):\n        super().__init__()\n        self.commits = 0\n    def commit(self):\n        self.commits += 1\nDAY = [D(2026, 11, 2)]\ndef make():\n    DAY[0] = D(2026, 11, 2)\n    return Library(_Counting(), lambda: DAY[0])\ndef refused(call, error, message):\n    try:\n        call()\n    except error as err:\n        assert str(err) == message, f\"got {str(err)!r}, expected {message!r}\"\n        return\n    raise AssertionError(f\"expected {error.__name__}: {message}\")\n\ndef stocked():\n    lib = make()\n    lib.add_book(\"978-0-14-118776-1\", \"Things Fall Apart\", \"Chinua Achebe\")\n    lib.add_book(\"9780435950828\", \"Arrow of God\", \"Chinua Achebe\")\n    lib.add_book(\"978-1-84749-058-2\", \"Half of a Yellow Sun\", \"Chimamanda Ngozi Adichie\")\n    lib.join(\"Ada Obi\")\n    lib.join(\"Bo Lee\")\n    lib.join(\"Chi Eze\")\n    return lib\n\ndef test_reserve_pay():\n    \"reserving and paying\"\n    lib = stocked()\n    lib.lend(\"B1\", \"M1\")\n    assert lib.reserve(\"B1\", \"M2\") == 1 and lib.reserve(\"b1\", \"m3\") == 2\n    refused(lambda: lib.reserve(\"B2\", \"M2\"), NotAllowed, \"B2 is on the shelf: borrow it instead\")\n    lib.member(\"M1\").owes = 80\n    assert lib.pay(\"M1\", \"\u00a30.30\") == 50 and lib.member(\"M1\").owes == 50\n    refused(lambda: lib.pay(\"M1\", \"lots\"), InvalidInput, \"not an amount: 'lots'\")\n    assert lib.store.commits == 10, \"commit after reserve and pay\"\ndef test_overdue():\n    \"overdue\"\n    lib = stocked()\n    lib.lend(\"B2\", \"M2\")\n    lib.lend(\"B1\", \"M1\")\n    DAY[0] = D(2026, 11, 10)\n    lib.lend(\"B3\", \"M3\")\n    DAY[0] = D(2026, 11, 25)\n    assert lib.overdue() == [(lib.store.loans[1], 2), (lib.store.loans[0], 2)], \"most days late first, then by book id\"\n    DAY[0] = D(2026, 12, 3)\n    assert [(l.book_id, d) for l, d in lib.overdue()] == [(\"B1\", 10), (\"B2\", 10), (\"B3\", 2)]\n    lib.give_back(\"B2\")\n    assert [l.book_id for l, d in lib.overdue()] == [\"B1\", \"B3\"]\ndef test_find():\n    \"finding\"\n    lib = stocked()\n    for n in range(7):\n        lib.add_book(\"9780141187761\", f\"Copy {n}\", \"Chinua Achebe\")\n    assert [b.book_id for b in lib.find(\"ACHEBE\")] == [\"B1\", \"B2\", \"B4\", \"B5\", \"B6\", \"B7\", \"B8\", \"B9\", \"B10\"]\n    assert [b.title for b in lib.find(\"yellow  sun\")] == [\"Half of a Yellow Sun\"]\n    assert [b.title for b in lib.find(\"fall achebe\")] == [\"Things Fall Apart\"] and lib.find(\"fall adichie\") == [] and len(lib.find(\"\")) == 10")

### Exercise: Step 22: describing results

The service (in `service.py`) returns objects; the interface turns them into text. Write `execute(command, args, library)`, which carries out one command with its list of arguments and returns the reply:

| Command | Reply |
|---|---|
| `add ISBN TITLE AUTHOR` | `added B3: Purple Hibiscus by Chimamanda Ngozi Adichie` |
| `join NAME` | `joined M3: Chi Eze` |
| `lend BOOK MEMBER` | `B1 lent to Ada Obi, due 2026-11-23` |
| `return BOOK` | `B1 returned`, or `B1 returned 3 days late: fine £0.60`; then, if someone is waiting, `; keep it for Bo Lee` |
| `reserve BOOK MEMBER` | `Bo Lee is number 1 in the queue for B1` |
| `pay MEMBER AMOUNT` | `Ada Obi now owes £0.30` |
| `overdue` | one line per late loan, `B1 Things Fall Apart: Ada Obi, 3 days late`, or `nothing is overdue` |
| `find WORDS...` | one line per book, from `describe()`, with ` (on loan)` added if it's out; or `no books found` |

Errors aren't caught here: the next step's `handle` does that. (The function can't be called `run`: the checks use that name.)

In [ ]:
# Run this cell first: it creates the files used below.
with open('service.py', 'w', encoding='utf-8') as f:
    f.write('"""The library\'s use cases. Takes and returns values; never prints."""\nimport logging\n\nfrom domain import Book, Member, check_can_borrow, give_back, lend, pay, reserve\nfrom values import check_all, clean_isbn, clean_text, parse_id, parse_pence\n\nlog = logging.getLogger("shelf.service")\n\n\nclass Library:\n    def __init__(self, store, today):\n        self.store = store\n        self.today = today          # a function returning today\'s date\n\n    def book(self, book_id):\n        return self.store.book(parse_id(book_id, "B"))\n\n    def member(self, member_id):\n        return self.store.member(parse_id(member_id, "M"))\n\n    def add_book(self, isbn, title, author):\n        v = check_all(isbn=lambda: clean_isbn(isbn),\n                      title=lambda: clean_text(title, "title", 100),\n                      author=lambda: clean_text(author, "author", 60))\n        book = Book(self.store.next_id("B"), v["isbn"], v["title"], v["author"])\n        self.store.add_book(book)\n        self.store.commit()\n        log.info("added %s", book.book_id)\n        return book\n\n    def join(self, name):\n        member = Member(self.store.next_id("M"), clean_text(name, "name", 60))\n        self.store.add_member(member)\n        self.store.commit()\n        log.info("added %s", member.member_id)\n        return member\n\n    def lend(self, book_id, member_id):\n        book, member = self.book(book_id), self.member(member_id)\n        open_loans, today = self.store.open_loans(member.member_id), self.today()\n        check_can_borrow(member, book, open_loans, today)    # first, so a refusal uses up no loan id\n        loan = lend(book, member, open_loans, today, self.store.next_id("L"))\n        self.store.add_loan(loan)\n        self.store.commit()\n        log.info("lent %s to %s", book.book_id, member.member_id)\n        return loan\n\n    def give_back(self, book_id):\n        book = self.book(book_id)\n        loan = self.store.open_loan_for(book.book_id)\n        fine = give_back(book, loan, self.store.member(loan.member_id), self.today())\n        self.store.commit()\n        log.info("%s returned, fine %d", book.book_id, fine)\n        return loan, fine\n\n    def reserve(self, book_id, member_id):\n        place = reserve(self.book(book_id), self.member(member_id))\n        self.store.commit()\n        return place\n\n    def pay(self, member_id, amount):\n        left = pay(self.member(member_id), parse_pence(amount))\n        self.store.commit()\n        return left\n\n    def overdue(self):\n        """(loan, days late) for every late loan, most days late first, then by book id."""\n        today = self.today()\n        late = [(loan, loan.days_late(today)) for loan in self.store.open_loans() if loan.days_late(today)]\n        return sorted(late, key=lambda pair: (-pair[1], pair[0].book_id))\n\n    def find(self, words):\n        """Books whose title or author contains every word (in any case), by id number."""\n        wanted = words.lower().split()\n        found = [b for b in self.store.books.values()\n                 if all(w in f"{b.title} {b.author}".lower() for w in wanted)]\n        return sorted(found, key=lambda b: int(b.book_id[1:]))\n')

In [ ]:
import shlex

from errors import LibraryError
from values import format_pence

USAGE = {
    "add": ("ISBN TITLE AUTHOR", 3),
    "join": ("NAME", 1),
    "lend": ("BOOK MEMBER", 2),
    "return": ("BOOK", 1),
    "reserve": ("BOOK MEMBER", 2),
    "pay": ("MEMBER AMOUNT", 2),
    "overdue": ("", 0),
    "find": ("WORDS", None),
}

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nfrom errors import NotAllowed\nfrom store import MemoryStore as _Mem\nfrom service import Library as _Lib\nD = _dt.date\nDAY = [D(2026, 11, 2)]\ndef make():\n    DAY[0] = D(2026, 11, 2)\n    lib = _Lib(_Mem(), lambda: DAY[0])\n    lib.add_book(\"978-0-14-118776-1\", \"Things Fall Apart\", \"Chinua Achebe\")\n    lib.add_book(\"9780435950828\", \"Arrow of God\", \"Chinua Achebe\")\n    lib.join(\"Ada Obi\")\n    lib.join(\"Bo Lee\")\n    return lib\n\ndef test_adding():\n    \"add and join\"\n    lib = make()\n    assert execute(\"add\", [\"9781847490582\", \"Half of a Yellow Sun\", \"Chimamanda Ngozi Adichie\"], lib) == \"added B3: Half of a Yellow Sun by Chimamanda Ngozi Adichie\"\n    assert execute(\"join\", [\"Chi Eze\"], lib) == \"joined M3: Chi Eze\"\ndef test_loans():\n    \"lend, reserve, return, pay\"\n    lib = make()\n    assert execute(\"lend\", [\"b1\", \"m1\"], lib) == \"B1 lent to Ada Obi, due 2026-11-23\"\n    assert execute(\"reserve\", [\"b1\", \"m2\"], lib) == \"Bo Lee is number 1 in the queue for B1\"\n    assert execute(\"lend\", [\"B2\", \"M1\"], lib) == \"B2 lent to Ada Obi, due 2026-11-23\"\n    assert execute(\"return\", [\"B2\"], lib) == \"B2 returned\"\n    DAY[0] = D(2026, 11, 26)\n    assert execute(\"overdue\", [], lib) == \"B1 Things Fall Apart: Ada Obi, 3 days late\"\n    assert execute(\"return\", [\"B1\"], lib) == \"B1 returned 3 days late: fine \u00a30.60; keep it for Bo Lee\"\n    assert execute(\"overdue\", [], lib) == \"nothing is overdue\"\n    assert execute(\"pay\", [\"m1\", \"0.3\"], lib) == \"Ada Obi now owes \u00a30.30\"\n    try:\n        execute(\"lend\", [\"B1\", \"M1\"], lib)\n        assert False, \"errors go through to handle\"\n    except NotAllowed:\n        pass\ndef test_find():\n    \"find\"\n    lib = make()\n    execute(\"lend\", [\"B2\", \"M2\"], lib)\n    assert execute(\"find\", [\"achebe\"], lib) == \"B1: Things Fall Apart by Chinua Achebe\\nB2: Arrow of God by Chinua Achebe (on loan)\"\n    assert execute(\"find\", [\"arrow\", \"god\"], lib) == \"B2: Arrow of God by Chinua Achebe (on loan)\"\n    assert execute(\"find\", [\"dickens\"], lib) == \"no books found\"")

### Exercise: Step 23: reading a line

`execute` is given. Write `handle(line, library)`, which turns one typed line into a reply:

1. Split the line with `shlex.split` (Week 21), so quoted titles stay together. An unclosed quote makes it raise `ValueError`: reply `"error: "` and its message.
2. A blank line gets the reply `""`.
3. The first word, in lower case, is the command. If it isn't in `USAGE`, reply `"unknown command: WORD"`.
4. `USAGE` gives each command's shape and how many arguments it takes (`None` means one or more). With the wrong number, reply `"usage: COMMAND SHAPE"` (with no space at the end when the shape is empty).
5. Otherwise return `execute`'s reply, turning any `LibraryError` into `"error: "` and its message. Nothing else is caught.

In [ ]:
# Run this cell first: it creates the files used below.
with open('service.py', 'w', encoding='utf-8') as f:
    f.write('"""The library\'s use cases. Takes and returns values; never prints."""\nimport logging\n\nfrom domain import Book, Member, check_can_borrow, give_back, lend, pay, reserve\nfrom values import check_all, clean_isbn, clean_text, parse_id, parse_pence\n\nlog = logging.getLogger("shelf.service")\n\n\nclass Library:\n    def __init__(self, store, today):\n        self.store = store\n        self.today = today          # a function returning today\'s date\n\n    def book(self, book_id):\n        return self.store.book(parse_id(book_id, "B"))\n\n    def member(self, member_id):\n        return self.store.member(parse_id(member_id, "M"))\n\n    def add_book(self, isbn, title, author):\n        v = check_all(isbn=lambda: clean_isbn(isbn),\n                      title=lambda: clean_text(title, "title", 100),\n                      author=lambda: clean_text(author, "author", 60))\n        book = Book(self.store.next_id("B"), v["isbn"], v["title"], v["author"])\n        self.store.add_book(book)\n        self.store.commit()\n        log.info("added %s", book.book_id)\n        return book\n\n    def join(self, name):\n        member = Member(self.store.next_id("M"), clean_text(name, "name", 60))\n        self.store.add_member(member)\n        self.store.commit()\n        log.info("added %s", member.member_id)\n        return member\n\n    def lend(self, book_id, member_id):\n        book, member = self.book(book_id), self.member(member_id)\n        open_loans, today = self.store.open_loans(member.member_id), self.today()\n        check_can_borrow(member, book, open_loans, today)    # first, so a refusal uses up no loan id\n        loan = lend(book, member, open_loans, today, self.store.next_id("L"))\n        self.store.add_loan(loan)\n        self.store.commit()\n        log.info("lent %s to %s", book.book_id, member.member_id)\n        return loan\n\n    def give_back(self, book_id):\n        book = self.book(book_id)\n        loan = self.store.open_loan_for(book.book_id)\n        fine = give_back(book, loan, self.store.member(loan.member_id), self.today())\n        self.store.commit()\n        log.info("%s returned, fine %d", book.book_id, fine)\n        return loan, fine\n\n    def reserve(self, book_id, member_id):\n        place = reserve(self.book(book_id), self.member(member_id))\n        self.store.commit()\n        return place\n\n    def pay(self, member_id, amount):\n        left = pay(self.member(member_id), parse_pence(amount))\n        self.store.commit()\n        return left\n\n    def overdue(self):\n        """(loan, days late) for every late loan, most days late first, then by book id."""\n        today = self.today()\n        late = [(loan, loan.days_late(today)) for loan in self.store.open_loans() if loan.days_late(today)]\n        return sorted(late, key=lambda pair: (-pair[1], pair[0].book_id))\n\n    def find(self, words):\n        """Books whose title or author contains every word (in any case), by id number."""\n        wanted = words.lower().split()\n        found = [b for b in self.store.books.values()\n                 if all(w in f"{b.title} {b.author}".lower() for w in wanted)]\n        return sorted(found, key=lambda b: int(b.book_id[1:]))\n')

In [ ]:
import shlex

from errors import LibraryError
from values import format_pence

USAGE = {
    "add": ("ISBN TITLE AUTHOR", 3),
    "join": ("NAME", 1),
    "lend": ("BOOK MEMBER", 2),
    "return": ("BOOK", 1),
    "reserve": ("BOOK MEMBER", 2),
    "pay": ("MEMBER AMOUNT", 2),
    "overdue": ("", 0),
    "find": ("WORDS", None),
}


def handle(line, library):
    ___


def execute(command, args, library):
    """Carry out one command, and describe the result."""
    if command == "add":
        return f"added {library.add_book(*args).describe()}"
    if command == "join":
        member = library.join(args[0])
        return f"joined {member.member_id}: {member.name}"
    if command == "lend":
        loan = library.lend(*args)
        return f"{loan.book_id} lent to {library.member(loan.member_id).name}, due {loan.due.isoformat()}"
    if command == "return":
        loan, fine = library.give_back(args[0])
        text = f"{loan.book_id} returned"
        if fine:
            text += f" {loan.days_late(loan.returned)} days late: fine {format_pence(fine)}"
        holds = library.book(loan.book_id).holds
        if holds:
            text += f"; keep it for {library.member(holds[0]).name}"
        return text
    if command == "reserve":
        place = library.reserve(*args)
        book, member = library.book(args[0]), library.member(args[1])
        return f"{member.name} is number {place} in the queue for {book.book_id}"
    if command == "pay":
        left = library.pay(*args)
        return f"{library.member(args[0]).name} now owes {format_pence(left)}"
    if command == "overdue":
        late = library.overdue()
        if not late:
            return "nothing is overdue"
        return "\n".join(f"{loan.book_id} {library.book(loan.book_id).title}: "
                         f"{library.member(loan.member_id).name}, {days} days late" for loan, days in late)
    if command == "find":
        books = library.find(" ".join(args))
        if not books:
            return "no books found"
        return "\n".join(b.describe() + (" (on loan)" if b.on_loan else "") for b in books)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nfrom errors import NotAllowed\nfrom store import MemoryStore as _Mem\nfrom service import Library as _Lib\nD = _dt.date\nDAY = [D(2026, 11, 2)]\ndef make():\n    DAY[0] = D(2026, 11, 2)\n    lib = _Lib(_Mem(), lambda: DAY[0])\n    lib.add_book(\"978-0-14-118776-1\", \"Things Fall Apart\", \"Chinua Achebe\")\n    lib.add_book(\"9780435950828\", \"Arrow of God\", \"Chinua Achebe\")\n    lib.join(\"Ada Obi\")\n    lib.join(\"Bo Lee\")\n    return lib\n\ndef test_lines():\n    \"lines\"\n    lib = make()\n    assert handle('add 978-1-84749-058-2 \"Half of a Yellow Sun\" \"Chimamanda Ngozi Adichie\"', lib) == \"added B3: Half of a Yellow Sun by Chimamanda Ngozi Adichie\"\n    assert handle(\"  LEND b1 m1 \", lib) == \"B1 lent to Ada Obi, due 2026-11-23\"\n    assert handle(\"\", lib) == \"\" and handle(\"   \", lib) == \"\"\ndef test_problems():\n    \"problems\"\n    lib = make()\n    assert handle(\"borrow B1 M1\", lib) == \"unknown command: borrow\"\n    assert handle(\"lend B1\", lib) == \"usage: lend BOOK MEMBER\"\n    assert handle(\"overdue now\", lib) == \"usage: overdue\"\n    assert handle(\"find\", lib) == \"usage: find WORDS\"\n    assert handle('join \"Ada', lib) == \"error: No closing quotation\"\n    assert handle(\"lend B9 M1\", lib) == \"error: no book B9\"\n    assert handle(\"lend B1 Ada\", lib) == \"error: not a member id: 'Ada'\"\n    assert handle('add 123 \"\" X', lib) == \"error: an ISBN has 13 digits: '123'; title is missing\"\ndef test_bugs():\n    \"bugs aren't hidden\"\n    class Broken:\n        def overdue(self):\n            raise TypeError(\"a bug\")\n    try:\n        handle(\"overdue\", Broken())\n        assert False, \"only LibraryError should be caught\"\n    except TypeError:\n        pass")

### Exercise: Step 24: wiring it together

Write the composition root:

- `build(env, today=date.today)` returns a `Library`, wired from the settings in `env` (a dictionary, like `os.environ`). `SHELF_LOG` names the logging level (default `"WARNING"`, any case; an unknown name means `WARNING`): call `logging.basicConfig` with that level, the format `"%(levelname)s %(name)s: %(message)s"` and `stream=sys.stderr`. `SHELF_DATA` is the data file (default `"library.json"`); the special value `":memory:"` means a `MemoryStore` instead of a `JsonStore`.
- `main(lines, env, today=date.today)` builds a library and prints the reply to each line, skipping empty replies.

In [ ]:
# Run this cell first: it creates the files used below.
with open('service.py', 'w', encoding='utf-8') as f:
    f.write('"""The library\'s use cases. Takes and returns values; never prints."""\nimport logging\n\nfrom domain import Book, Member, check_can_borrow, give_back, lend, pay, reserve\nfrom values import check_all, clean_isbn, clean_text, parse_id, parse_pence\n\nlog = logging.getLogger("shelf.service")\n\n\nclass Library:\n    def __init__(self, store, today):\n        self.store = store\n        self.today = today          # a function returning today\'s date\n\n    def book(self, book_id):\n        return self.store.book(parse_id(book_id, "B"))\n\n    def member(self, member_id):\n        return self.store.member(parse_id(member_id, "M"))\n\n    def add_book(self, isbn, title, author):\n        v = check_all(isbn=lambda: clean_isbn(isbn),\n                      title=lambda: clean_text(title, "title", 100),\n                      author=lambda: clean_text(author, "author", 60))\n        book = Book(self.store.next_id("B"), v["isbn"], v["title"], v["author"])\n        self.store.add_book(book)\n        self.store.commit()\n        log.info("added %s", book.book_id)\n        return book\n\n    def join(self, name):\n        member = Member(self.store.next_id("M"), clean_text(name, "name", 60))\n        self.store.add_member(member)\n        self.store.commit()\n        log.info("added %s", member.member_id)\n        return member\n\n    def lend(self, book_id, member_id):\n        book, member = self.book(book_id), self.member(member_id)\n        open_loans, today = self.store.open_loans(member.member_id), self.today()\n        check_can_borrow(member, book, open_loans, today)    # first, so a refusal uses up no loan id\n        loan = lend(book, member, open_loans, today, self.store.next_id("L"))\n        self.store.add_loan(loan)\n        self.store.commit()\n        log.info("lent %s to %s", book.book_id, member.member_id)\n        return loan\n\n    def give_back(self, book_id):\n        book = self.book(book_id)\n        loan = self.store.open_loan_for(book.book_id)\n        fine = give_back(book, loan, self.store.member(loan.member_id), self.today())\n        self.store.commit()\n        log.info("%s returned, fine %d", book.book_id, fine)\n        return loan, fine\n\n    def reserve(self, book_id, member_id):\n        place = reserve(self.book(book_id), self.member(member_id))\n        self.store.commit()\n        return place\n\n    def pay(self, member_id, amount):\n        left = pay(self.member(member_id), parse_pence(amount))\n        self.store.commit()\n        return left\n\n    def overdue(self):\n        """(loan, days late) for every late loan, most days late first, then by book id."""\n        today = self.today()\n        late = [(loan, loan.days_late(today)) for loan in self.store.open_loans() if loan.days_late(today)]\n        return sorted(late, key=lambda pair: (-pair[1], pair[0].book_id))\n\n    def find(self, words):\n        """Books whose title or author contains every word (in any case), by id number."""\n        wanted = words.lower().split()\n        found = [b for b in self.store.books.values()\n                 if all(w in f"{b.title} {b.author}".lower() for w in wanted)]\n        return sorted(found, key=lambda b: int(b.book_id[1:]))\n')
with open('cli.py', 'w', encoding='utf-8') as f:
    f.write('"""The text interface: turns typed lines into calls, and results into text."""\nimport shlex\n\nfrom errors import LibraryError\nfrom values import format_pence\n\nUSAGE = {\n    "add": ("ISBN TITLE AUTHOR", 3),\n    "join": ("NAME", 1),\n    "lend": ("BOOK MEMBER", 2),\n    "return": ("BOOK", 1),\n    "reserve": ("BOOK MEMBER", 2),\n    "pay": ("MEMBER AMOUNT", 2),\n    "overdue": ("", 0),\n    "find": ("WORDS", None),\n}\n\n\ndef handle(line, library):\n    """One typed line in, one piece of text out."""\n    try:\n        words = shlex.split(line)\n    except ValueError as err:\n        return f"error: {err}"\n    if not words:\n        return ""\n    command, args = words[0].lower(), words[1:]\n    if command not in USAGE:\n        return f"unknown command: {command}"\n    shape, count = USAGE[command]\n    if (count is None and not args) or (count is not None and len(args) != count):\n        return f"usage: {command} {shape}".rstrip()\n    try:\n        return execute(command, args, library)\n    except LibraryError as err:\n        return f"error: {err}"\n\n\ndef execute(command, args, library):\n    """Carry out one command, and describe the result."""\n    if command == "add":\n        return f"added {library.add_book(*args).describe()}"\n    if command == "join":\n        member = library.join(args[0])\n        return f"joined {member.member_id}: {member.name}"\n    if command == "lend":\n        loan = library.lend(*args)\n        return f"{loan.book_id} lent to {library.member(loan.member_id).name}, due {loan.due.isoformat()}"\n    if command == "return":\n        loan, fine = library.give_back(args[0])\n        text = f"{loan.book_id} returned"\n        if fine:\n            text += f" {loan.days_late(loan.returned)} days late: fine {format_pence(fine)}"\n        holds = library.book(loan.book_id).holds\n        if holds:\n            text += f"; keep it for {library.member(holds[0]).name}"\n        return text\n    if command == "reserve":\n        place = library.reserve(*args)\n        book, member = library.book(args[0]), library.member(args[1])\n        return f"{member.name} is number {place} in the queue for {book.book_id}"\n    if command == "pay":\n        left = library.pay(*args)\n        return f"{library.member(args[0]).name} now owes {format_pence(left)}"\n    if command == "overdue":\n        late = library.overdue()\n        if not late:\n            return "nothing is overdue"\n        return "\\n".join(f"{loan.book_id} {library.book(loan.book_id).title}: "\n                         f"{library.member(loan.member_id).name}, {days} days late" for loan, days in late)\n    if command == "find":\n        books = library.find(" ".join(args))\n        if not books:\n            return "no books found"\n        return "\\n".join(b.describe() + (" (on loan)" if b.on_loan else "") for b in books)\n')

In [ ]:
"""The composition root: builds the program from settings, and runs it."""
import logging
import sys
from datetime import date

from cli import handle
from service import Library
from store import JsonStore, MemoryStore

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nimport logging as _logging\nimport json as _json\nD = _dt.date\ndef test_build():\n    \"building\"\n    lib = build({\"SHELF_DATA\": \":memory:\"}, lambda: D(2026, 11, 2))\n    from store import MemoryStore, JsonStore\n    assert type(lib.store) is MemoryStore and lib.today() == D(2026, 11, 2)\n    lib = build({})\n    assert type(lib.store) is JsonStore and lib.store.path == \"library.json\" and lib.today() == _dt.date.today()\n    assert type(build({\"SHELF_DATA\": \"other.json\"}).store) is JsonStore\ndef test_logging():\n    \"the log level\"\n    for value, level in [({\"SHELF_LOG\": \"debug\"}, _logging.DEBUG), ({}, _logging.WARNING), ({\"SHELF_LOG\": \"loud\"}, _logging.WARNING), ({\"SHELF_LOG\": \"Info\"}, _logging.INFO)]:\n        root = _logging.getLogger()\n        for h in list(root.handlers):\n            root.removeHandler(h)\n        build({\"SHELF_DATA\": \":memory:\", **value})\n        assert root.level == level, f\"{value}: level {root.level}\"\ndef test_main():\n    \"a session, saved between runs\"\n    import io, contextlib\n    reset_files()\n    buf = io.StringIO()\n    with contextlib.redirect_stdout(buf):\n        main(['add 978-0-14-118776-1 \"Things Fall Apart\" \"Chinua Achebe\"', \"\", 'join \"Ada Obi\"', \"lend B1 M1\"], {\"SHELF_DATA\": \"lib.json\"}, lambda: D(2026, 11, 2))\n        main([\"return B1\", \"find fall\"], {\"SHELF_DATA\": \"lib.json\"}, lambda: D(2026, 11, 26))\n    assert buf.getvalue().splitlines() == [\"added B1: Things Fall Apart by Chinua Achebe\", \"joined M1: Ada Obi\", \"B1 lent to Ada Obi, due 2026-11-23\",\n                                           \"B1 returned 3 days late: fine \u00a30.60\", \"B1: Things Fall Apart by Chinua Achebe\"], buf.getvalue()\n    assert _json.loads(read_file(\"lib.json\"))[\"members\"][0][\"owes\"] == 60")

In [ ]:
from store import JsonStore, MemoryStore
import logging

from domain import Book, Member, check_can_borrow, give_back, lend, pay, reserve
from values import check_all, clean_isbn, clean_text, parse_id, parse_pence

log = logging.getLogger("shelf.service")


class Library:
    def __init__(self, store, today):
        self.store = store
        self.today = today          # a function returning today's date

    def book(self, book_id):
        return self.store.book(parse_id(book_id, "B"))

    def member(self, member_id):
        return self.store.member(parse_id(member_id, "M"))

    def add_book(self, isbn, title, author):
        v = check_all(isbn=lambda: clean_isbn(isbn),
                      title=lambda: clean_text(title, "title", 100),
                      author=lambda: clean_text(author, "author", 60))
        book = Book(self.store.next_id("B"), v["isbn"], v["title"], v["author"])
        self.store.add_book(book)
        self.store.commit()
        log.info("added %s", book.book_id)
        return book

    def join(self, name):
        member = Member(self.store.next_id("M"), clean_text(name, "name", 60))
        self.store.add_member(member)
        self.store.commit()
        log.info("added %s", member.member_id)
        return member

    def lend(self, book_id, member_id):
        book, member = self.book(book_id), self.member(member_id)
        open_loans, today = self.store.open_loans(member.member_id), self.today()
        check_can_borrow(member, book, open_loans, today)    # first, so a refusal uses up no loan id
        loan = lend(book, member, open_loans, today, self.store.next_id("L"))
        self.store.add_loan(loan)
        self.store.commit()
        log.info("lent %s to %s", book.book_id, member.member_id)
        return loan

    def give_back(self, book_id):
        book = self.book(book_id)
        loan = self.store.open_loan_for(book.book_id)
        fine = give_back(book, loan, self.store.member(loan.member_id), self.today())
        self.store.commit()
        log.info("%s returned, fine %d", book.book_id, fine)
        return loan, fine

    def reserve(self, book_id, member_id):
        place = reserve(self.book(book_id), self.member(member_id))
        self.store.commit()
        return place

    def pay(self, member_id, amount):
        left = pay(self.member(member_id), parse_pence(amount))
        self.store.commit()
        return left

    def overdue(self):
        """(loan, days late) for every late loan, most days late first, then by book id."""
        today = self.today()
        late = [(loan, loan.days_late(today)) for loan in self.store.open_loans() if loan.days_late(today)]
        return sorted(late, key=lambda pair: (-pair[1], pair[0].book_id))

    def find(self, words):
        """Books whose title or author contains every word (in any case), by id number."""
        wanted = words.lower().split()
        found = [b for b in self.store.books.values()
                 if all(w in f"{b.title} {b.author}".lower() for w in wanted)]
        return sorted(found, key=lambda b: int(b.book_id[1:]))


# ---- cli.py ----
import shlex

from errors import LibraryError
from values import format_pence

USAGE = {
    "add": ("ISBN TITLE AUTHOR", 3),
    "join": ("NAME", 1),
    "lend": ("BOOK MEMBER", 2),
    "return": ("BOOK", 1),
    "reserve": ("BOOK MEMBER", 2),
    "pay": ("MEMBER AMOUNT", 2),
    "overdue": ("", 0),
    "find": ("WORDS", None),
}


def handle(line, library):
    """One typed line in, one piece of text out."""
    try:
        words = shlex.split(line)
    except ValueError as err:
        return f"error: {err}"
    if not words:
        return ""
    command, args = words[0].lower(), words[1:]
    if command not in USAGE:
        return f"unknown command: {command}"
    shape, count = USAGE[command]
    if (count is None and not args) or (count is not None and len(args) != count):
        return f"usage: {command} {shape}".rstrip()
    try:
        return execute(command, args, library)
    except LibraryError as err:
        return f"error: {err}"


def execute(command, args, library):
    """Carry out one command, and describe the result."""
    if command == "add":
        return f"added {library.add_book(*args).describe()}"
    if command == "join":
        member = library.join(args[0])
        return f"joined {member.member_id}: {member.name}"
    if command == "lend":
        loan = library.lend(*args)
        return f"{loan.book_id} lent to {library.member(loan.member_id).name}, due {loan.due.isoformat()}"
    if command == "return":
        loan, fine = library.give_back(args[0])
        text = f"{loan.book_id} returned"
        if fine:
            text += f" {loan.days_late(loan.returned)} days late: fine {format_pence(fine)}"
        holds = library.book(loan.book_id).holds
        if holds:
            text += f"; keep it for {library.member(holds[0]).name}"
        return text
    if command == "reserve":
        place = library.reserve(*args)
        book, member = library.book(args[0]), library.member(args[1])
        return f"{member.name} is number {place} in the queue for {book.book_id}"
    if command == "pay":
        left = library.pay(*args)
        return f"{library.member(args[0]).name} now owes {format_pence(left)}"
    if command == "overdue":
        late = library.overdue()
        if not late:
            return "nothing is overdue"
        return "\n".join(f"{loan.book_id} {library.book(loan.book_id).title}: "
                         f"{library.member(loan.member_id).name}, {days} days late" for loan, days in late)
    if command == "find":
        books = library.find(" ".join(args))
        if not books:
            return "no books found"
        return "\n".join(b.describe() + (" (on loan)" if b.on_loan else "") for b in books)


# ---- app.py ----
import logging
import sys
from datetime import date



def build(env, today=date.today):
    """A Library wired up from settings (a dictionary such as os.environ)."""
    level = env.get("SHELF_LOG", "WARNING").upper()
    logging.basicConfig(level=getattr(logging, level, logging.WARNING),
                        format="%(levelname)s %(name)s: %(message)s", stream=sys.stderr)
    path = env.get("SHELF_DATA", "library.json")
    store = MemoryStore() if path == ":memory:" else JsonStore(path)
    return Library(store, today)


def main(lines, env, today=date.today):
    library = build(env, today)
    for line in lines:
        reply = handle(line, library)
        if reply:
            print(reply)

day = [date(2026, 11, 2)]
settings = {"SHELF_DATA": "library.json"}
main([
    'add 978-0-14-118776-1 "Things Fall Apart" "Chinua Achebe"',
    'add 9780435950828 "Arrow of God" "Chinua Achebe"',
    'join "Ada Obi"',
    'join "Bo Lee"',
    "lend B1 M1",
    "lend b1 m2",
    "reserve B1 M2",
    "find achebe",
], settings, lambda: day[0])

print("--- three and a half weeks later, the program is started again ---")
day[0] = date(2026, 11, 26)
main(["overdue", "lend B2 M1", "return B1", "lend B1 M1", "lend B1 M2", "pay M1 £0.60", "lend B2 M1", "frob"], settings, lambda: day[0])

> **One line, five layers.** Follow `lend b1 m2` through the program. `handle` splits it and checks the number of arguments; `execute` calls `library.lend("b1", "m2")`; the service reads the ids into `"B1"` and `"M2"`, finds the book and member in the store, and asks the domain's `lend`; the domain's `check_can_borrow` finds the book on loan and raises `NotAllowed("B1 is on loan")`; nothing commits; the exception passes back up through the service and `execute`, untouched, until `handle` turns it into `error: B1 is on loan`. Each layer did one job, and only the top one decided what the volunteer sees.

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Where we are

The modules so far are in the files below: `errors.py`, `values.py`, `domain.py` and `store.py`. Today adds the last three:

| Module | Layer | Depends on |
|---|---|---|
| `service.py` | application | domain, values, and a store (passed in) |
| `cli.py` | presentation | the service (passed in), `errors`, `values.format_pence` |
| `app.py` | composition root | everything: it's the only module that does |

Each use case in the service follows the same four beats, which makes them short and easy to check:

1. **Read** the typed values (`parse_id`, `parse_pence`, `check_all`), raising `InvalidInput`.
2. **Find** the objects in the store, raising `NotFound`.
3. **Ask the domain** to do the work, which raises `NotAllowed` or changes the objects.
4. **Commit**, and log what happened.

The service never prints and never catches `LibraryError`: it passes refusals up to the interface, which decides how to show them (Week 37's "log once, where it's handled" applies to messages for people too).

## Step 19: books and members

## Step 20: lending and returning

## Step 21: queues, payments and questions

`overdue` and `find` are **queries**: they change nothing, so they don't commit (Week 28's commands and queries).

## Step 22: describing results

Now the presentation layer, `cli.py`. It's split in two: `execute` carries out one command and describes the result, and the next step's `handle` reads the line and deals with errors. Keeping them apart means each is short, and `execute` can be tested without thinking about parsing.

## Step 23: reading a line

Why catch `LibraryError` but nothing else? Every refusal the program makes on purpose is a `LibraryError` with a message written for people, so turning it into `error: ...` is always right. Anything else is a bug: catching it would show the volunteer a confusing message and hide the bug from whoever could fix it.

## Step 24: wiring it together

The composition root (Week 36) is the one place that knows which store, which clock and which log level the program uses. Everything else receives them.

## Trying it out

Here's the whole program: two sessions, as if a volunteer had closed `shelf` and opened it again three and a half weeks later. The data survives in `library.json` between them (open it after running). Change `SHELF_DATA` to `":memory:"`, or add `"SHELF_LOG": "info"` to the settings, to see the composition root at work.

## Quiz

## Before you go

Put today's code in `service.py`, `cli.py` and `app.py`. To use `shelf` for real, add a small loop at the bottom of `app.py`:

```python
if __name__ == "__main__":
    import os

    def lines():
        while True:
            try:
                yield input("> ")
            except EOFError:
                return

    main(lines(), os.environ)
```

(`yield` makes `lines` a generator, which Phase 5 explains; for now, it hands `main` one typed line at a time until you press Ctrl+D.) Run `python3 app.py`, lend yourself a book, and commit. Tomorrow: a proper test suite for every layer, and a refactoring made safe by it.